# DK-SOFNN: Data-Knowledge-Driven Self-Organizing Fuzzy Neural Network

**Paper:** H. Han, X. Liu, and J. Qiao, "Data-Knowledge-Driven Self-Organizing Fuzzy Neural Network," *IEEE Trans. Neural Netw. Learn. Syst.*, vol. 35, no. 2, Feb. 2024.

## Overview

This notebook demonstrates the full DK-SOFNN pipeline:

1. **Source FNN training** — A five-layer fuzzy neural network (Eqs. 1–5) is trained on a large source dataset using gradient descent (Eqs. 7–9).
2. **Structure knowledge mining** — Similarity ($R$), Sensitivity ($M$), and Contribution ($C$) indexes are computed (Eqs. 10–12).
3. **Source self-organization** — Rules are grown or pruned based on the knowledge indexes (Eqs. 13–15).
4. **Target FNN initialization** — The target FNN inherits the source FNN's optimized structure and parameters.
5. **DK-SOFNN target training** — Structure adjustment and parameter reinforcement with knowledge transfer (Table II, Eqs. 23–30).
6. **Evaluation** — RMSE (Eq. 42), sMAPE (Eq. 43), and MASE (Eq. 44).

## 1. Imports and Setup

We import the `DK_SOFNN` module, which contains the full implementation, plus `matplotlib` for visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import sys
import os

# Ensure the module directory is on the path
sys.path.insert(0, os.path.abspath('.'))

from DK_SOFNN import (
    FNN,
    train_source_fnn,
    calculate_indexes,
    source_self_organization,
    train_dk_sofnn,
    calculate_metrics,
    generate_mackey_glass,
    print_rule_base,
    SEED, INITIAL_RULES, SOURCE_EPOCHS, SOURCE_LR,
    TARGET_LR, N_T_DEFAULT, MIN_RULES, MAX_RULES,
)

np.random.seed(SEED)
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('All imports successful.')

## 2. Load and Preprocess Data

We attempt to load the **Combined Cycle Power Plant** dataset from `Folds5x2_pp.xlsx`. If the file is unavailable, we fall back to the **Mackey-Glass** chaotic time-series (a standard benchmark for FNN).

All features and targets are normalized to $[0, 1]$ using Min-Max scaling. The data is split into:
- **Source** — 1,500 samples for training the source FNN
- **Target train** — 100 samples (with added noise to simulate domain shift)
- **Target test** — 300 samples for final evaluation

In [ ]:
from sklearn.preprocessing import MinMaxScaler

feature_names = None
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
dataset_name = None

try:
    import pandas as pd
    df = pd.read_excel('Folds5x2_pp.xlsx')
    X = df[['AT', 'V', 'AP', 'RH']].values
    y = df['PE'].values.reshape(-1, 1)
    feature_names = ['AT', 'V', 'AP', 'RH']
    dataset_name = 'Combined Cycle Power Plant'
    print(f'Loaded Combined Cycle Power Plant: {X.shape[0]} samples, {X.shape[1]} features')
except Exception as e:
    print(f'Excel data not available ({e}). Using Mackey-Glass fallback.')
    X, y = generate_mackey_glass(n_samples=2000, seed=SEED)
    feature_names = ['lag1', 'lag2', 'lag3', 'lag4']
    dataset_name = 'Mackey-Glass (synthetic)'
    print(f'Generated Mackey-Glass: {X.shape[0]} samples, {X.shape[1]} features')

# Normalize to [0, 1]
X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y)

# Split: source / target-train / target-test
n_source = min(1500, len(X_norm) - 400)
n_target_train = 100
n_target_test = min(300, len(X_norm) - n_source - n_target_train)

X_source = X_norm[:n_source]
y_source = y_norm[:n_source]

X_target_train = X_norm[n_source:n_source + n_target_train]
y_target_train = y_norm[n_source:n_source + n_target_train]

X_target_test = X_norm[n_source + n_target_train:
                       n_source + n_target_train + n_target_test]
y_target_test = y_norm[n_source + n_target_train:
                       n_source + n_target_train + n_target_test]

# Domain shift: add Gaussian noise to target labels
noise = np.random.normal(0, 0.05, y_target_train.shape)
y_target_train_noisy = np.clip(y_target_train + noise, 0, 1)

n_inputs = X_source.shape[1]
print(f'\nDataset: {dataset_name}')
print(f'  Source samples:       {len(X_source)}')
print(f'  Target train samples: {len(X_target_train)} (with noise)')
print(f'  Target test samples:  {len(X_target_test)}')

## 3. Phase 1 — Train the Source FNN (Eqs. 7–9)

The five-layer FNN (Eqs. 1–5) is trained via stochastic gradient descent:

$$\Delta w_k = -\lambda_S \, e \, v_k \qquad \text{(Eq. 7)}$$

$$\Delta c_{k,p} = -\lambda_S \, e \, \frac{w_k - y}{\sum u} \, u_k \, \frac{x_p - c_{k,p}}{\sigma_{k,p}^2} \qquad \text{(Eq. 8)}$$

$$\Delta \sigma_{k,p} = -\lambda_S \, e \, \frac{w_k - y}{\sum u} \, u_k \, \frac{(x_p - c_{k,p})^2}{\sigma_{k,p}^3} \qquad \text{(Eq. 9)}$$

Parameters: $\lambda_S = 0.1$, $I = 300$ epochs, $K = 20$ initial rules.

In [ ]:
source_fnn = FNN(n_inputs=n_inputs, n_rules=INITIAL_RULES)
source_mse = train_source_fnn(source_fnn, X_source, y_source,
                              epochs=SOURCE_EPOCHS, lr=SOURCE_LR)

## 4. Phase 2 — Mine Structure Knowledge (Eqs. 10–12)

Three knowledge indexes are computed for each fuzzy rule:

- **Similarity** $R_k$ (Eq. 10) — overlap between rules
- **Sensitivity** $M_k$ (Eq. 11) — impact of rule $k$ on the output
- **Contribution** $C_k$ (Eq. 12) — effective contribution to accuracy

These indexes guide the self-organization of the network structure.

In [ ]:
R_s, M_s, C_s = calculate_indexes(source_fnn, X_source, y_source)
print(f'Similarity   R_S (per rule): mean={np.mean(R_s):.4f}, std={np.std(R_s):.4f}')
print(f'Sensitivity  M_S (per rule): mean={np.mean(M_s):.4f}, std={np.std(M_s):.4f}')
print(f'Contribution C_S (per rule): mean={np.mean(C_s):.4f}, std={np.std(C_s):.4f}')

## 5. Phase 3 — Self-Organize the Source FNN (Eqs. 13–15)

Rules are grown or pruned based on the mined knowledge indexes:

- **Grow** a rule if average indexes suggest under-coverage
- **Prune** a rule if it is highly similar and has low contribution

After self-organization, the indexes are recomputed and the expert rule (highest $C_k$) is identified for knowledge transfer.

In [ ]:
source_fnn = source_self_organization(source_fnn, X_source, y_source,
                                      n_iterations=5)

# Recompute indexes after self-organization
R_s, M_s, C_s = calculate_indexes(source_fnn, X_source, y_source)
source_avgs = (np.mean(R_s), np.mean(M_s), np.mean(C_s))
print(f'Post-self-org averages: R={source_avgs[0]:.4f}, '
      f'M={source_avgs[1]:.4f}, C={source_avgs[2]:.4f}')

# Identify the expert rule (highest contribution C)
best_rule_idx = np.argmax(C_s)
source_expert = {
    'center': source_fnn.centers[best_rule_idx].copy(),
    'width':  source_fnn.widths[best_rule_idx].copy(),
    'weight': source_fnn.weights[best_rule_idx].copy(),
}
source_params = {
    'centers': source_fnn.centers.copy(),
    'widths':  source_fnn.widths.copy(),
    'weights': source_fnn.weights.copy(),
}
print(f'Expert rule index: {best_rule_idx}')
print(f'Source FNN rules after self-organization: {source_fnn.n_rules}')

## 6. Phase 4 — Initialize the Target FNN (Knowledge Transfer)

The target FNN is initialized by copying the source FNN's optimized structure (centers $c_{k,p}$, widths $\sigma_{k,p}$, and consequent weights $w_k$). This transfers the learned knowledge to the target domain.

In [ ]:
target_fnn = FNN(n_inputs=n_inputs, n_rules=source_fnn.n_rules)
target_fnn.centers = source_fnn.centers.copy()
target_fnn.widths  = source_fnn.widths.copy()
target_fnn.weights = source_fnn.weights.copy()
print(f'Target FNN initialized with {target_fnn.n_rules} rules from source.')

## 7. Phase 5 — DK-SOFNN Target Training (Table II)

The full DK-SOFNN algorithm (Table II) trains the target FNN:

- **Structure adjustment** (every $N_T = 50$ samples): grow, prune, compensate, or keep rules constant (Steps 6–9).
- **Parameter reinforcement** (every sample): update parameters with $H$ learning frameworks using $(\alpha_h, \beta_h)$ pairs (Steps 10–13, Eqs. 23–30).

Target learning rate: $\lambda_T = 0.01$.

In [ ]:
rule_history, error_history = train_dk_sofnn(
    source_fnn, target_fnn,
    X_target_train, y_target_train_noisy,
    source_expert, source_avgs, source_params,
    N_T=N_T_DEFAULT, lr=TARGET_LR,
)

## 8. Phase 6 — Evaluation (Eqs. 42–44)

Performance metrics on the held-out target test set:

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2} \qquad \text{(Eq. 42)}$$

$$\text{sMAPE} = \frac{1}{N}\sum_{i=1}^{N} \frac{2|\hat{y}_i - y_i|}{|\hat{y}_i| + |y_i|} \qquad \text{(Eq. 43)}$$

$$\text{MASE} = \frac{1}{N}\sum_{i=1}^{N} \left(\frac{\hat{y}_i - y_i}{y_i}\right)^2 \qquad \text{(Eq. 44)}$$

In [ ]:
y_test_pred = target_fnn.predict(X_target_test).reshape(-1, 1)
rmse, smape, mase = calculate_metrics(y_target_test, y_test_pred)

print(f'Final rule count: {target_fnn.n_rules}')
print(f'RMSE  (Eq. 42): {rmse:.6f}')
print(f'sMAPE (Eq. 43): {smape:.6f}')
print(f'MASE  (Eq. 44): {mase:.6f}')

# Denormalized RMSE
y_test_real = scaler_y.inverse_transform(y_target_test)
y_pred_real = scaler_y.inverse_transform(y_test_pred)
rmse_real = np.sqrt(np.mean((y_test_real - y_pred_real) ** 2))
print(f'RMSE (original scale): {rmse_real:.4f}')

## 9. Visualization

Four plots summarizing the DK-SOFNN results:

1. **Training convergence** — Source MSE vs. epoch
2. **Structure adjustment** — Rule count vs. training step
3. **Prediction vs. actual** — Scatter plot on test set
4. **Error distribution** — Histogram of prediction errors

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'DK-SOFNN Results — {dataset_name}', fontsize=14, fontweight='bold')

# --- 1. Training convergence (source MSE vs epoch) ---
ax = axes[0, 0]
ax.semilogy(range(1, len(source_mse) + 1), source_mse, linewidth=1.2, color='#2563EB')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Source FNN Training Convergence')
ax.grid(True, alpha=0.3)

# --- 2. Structure adjustment (rule count vs training step) ---
ax = axes[0, 1]
ax.plot(range(len(rule_history)), rule_history, linewidth=1.2, color='#DC2626')
ax.set_xlabel('Training Step (sample index)')
ax.set_ylabel('Number of Rules')
ax.set_title('Target FNN Structure Adjustment')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# --- 3. Prediction vs actual scatter plot ---
ax = axes[1, 0]
yt = y_test_real.flatten()
yp = y_pred_real.flatten()
ax.scatter(yt, yp, alpha=0.5, s=15, color='#059669', edgecolors='none')
lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
margin = (lims[1] - lims[0]) * 0.05
ax.plot([lims[0]-margin, lims[1]+margin], [lims[0]-margin, lims[1]+margin],
        'k--', linewidth=0.8, label='Perfect prediction')
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title('Prediction vs. Actual (Test Set)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- 4. Error distribution histogram ---
ax = axes[1, 1]
errors = yp - yt
ax.hist(errors, bins=30, color='#7C3AED', edgecolor='white', alpha=0.8)
ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
ax.set_xlabel('Prediction Error')
ax.set_ylabel('Frequency')
ax.set_title(f'Error Distribution (mean={np.mean(errors):.4f}, std={np.std(errors):.4f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Final Fuzzy Rule Base

The learned IF-THEN rules in the target FNN after DK-SOFNN training. Each rule has the form:

> **IF** $x_1 \approx c_{k,1}$ **AND** $x_2 \approx c_{k,2}$ **AND** ... **THEN** $y = w_k$

Centers and weights are shown in the original (denormalized) scale.

In [ ]:
print_rule_base(target_fnn, feature_names, scaler_X, scaler_y)

print('\nDK-SOFNN pipeline complete.')